# Skin tone fairness: training-composition experiments

Trains and evaluates three models on Fitzpatrick17k-C, then compares their performance across skin types:

- **A**: trained on all skin types
- **B**: trained on light skin only (Fitzpatrick I-II)
- **C**: trained on all skin types, subsampled to the same size and diagnosis mix as B

**Steps**
1. *Runtime → Change runtime type → **T4 GPU** → Save*
2. If the dataset authors sent you a download link for the full image set, paste it into `FULL_IMAGES_URL` below.
   Without it, the notebook uses only the images that can still be downloaded from the original atlas
   websites (a small pilot; results will not be meaningful for darker skin).
3. *Runtime → **Run all***, and keep this tab open (about 60-75 minutes with the full dataset).

In [ ]:
FULL_IMAGES_URL = ""  # link from the Fitzpatrick17k authors (Google Drive, Dropbox or direct URL)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: choose Runtime → Change runtime type → T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Get the code and labels

In [ ]:
import os
os.chdir("/content")
!rm -rf skin-tone-fairness && git clone -q https://github.com/i-oladele/skin-tone-fairness.git
os.chdir("/content/skin-tone-fairness")
!pip install -q -e .
!mkdir -p data/raw
!curl -sSL -o data/raw/fitzpatrick17k.csv https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv
!curl -sSL -o data/raw/Fitzpatrick17k-C.csv "https://zenodo.org/api/records/12739457/files/Fitzpatrick17k-C.csv/content"
!wc -l data/raw/*.csv

## 2. Get the images (each is checked against its MD5 fingerprint)

In [ ]:
url = FULL_IMAGES_URL.strip()
if url:
    if "drive.google.com" in url:
        !gdown --fuzzy "{url}" -O data/raw/images_archive
    else:
        url = url.replace("dl=0", "dl=1")
        !curl -L --retry 5 -o data/raw/images_archive "{url}"
    !python scripts/download_images.py --archive data/raw/images_archive
else:
    print("No FULL_IMAGES_URL given: downloading what still exists at the original atlas links (pilot only).")
    !python scripts/download_images.py --workers 16

## 3. Describe the data and prepare the images

In [ ]:
!python scripts/explore_data.py
!python scripts/prepare_data.py

## 4. Train A: all skin types

In [ ]:
!python scripts/train.py --run-name all_types --num-workers 2

## 5. Train B: light skin only (I-II)

In [ ]:
!python scripts/train.py --run-name light_only --train-groups I-II --num-workers 2

## 6. Train C: all skin types, same size as B

In [ ]:
!python scripts/train.py --run-name size_matched --size-match I-II --num-workers 2

## 7. Evaluate all three, overall and by skin type

In [ ]:
!python scripts/evaluate.py --run runs/all_types --num-workers 2
!python scripts/evaluate.py --run runs/light_only --num-workers 2
!python scripts/evaluate.py --run runs/size_matched --num-workers 2

## 8. Compare

In [ ]:
!python scripts/compare_runs.py runs/all_types runs/light_only runs/size_matched --names "A: all skin types" "B: light skin only (I-II)" "C: all skin types, same size as B" --out runs/composition_comparison.md --figure runs/composition_comparison.png

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/data_overview.md").read()))
display(Image("runs/composition_comparison.png", width=1000))
display(Markdown(open("runs/composition_comparison.md").read()))
display(Markdown(open("runs/all_types/evaluation/report.md").read()))

## 9. Save the results

Colab deletes everything when the session ends. If the download doesn't start, open the **Files** panel
(folder icon on the left), find `skin_tone_results.zip` in `skin-tone-fairness`, and choose **⋮ → Download**.

In [ ]:
!cp results/data_overview.md runs/ && cp docs/figures/images_by_skin_type.png docs/figures/annotator_agreement.png runs/
!cp data/raw/download_log.csv runs/
!cd runs && zip -qr ../skin_tone_results.zip .
from google.colab import files
files.download("skin_tone_results.zip")